# 01 — Data cleaning

**Goal:** combine the six raw city files from NASA POWER into one clean daily dataset, ready for analysis.

**Input:** `data/raw/*.csv` (one file per city)

**Output:** `data/processed/madagascar_climate_daily.csv`

In [10]:
from pathlib import Path

import pandas as pd

RAW_DIR= Path("../data/raw")
PROCESSED_DIR= Path("../data/processed")

## 1. Load and combine the raw files
Each city has its own file. We stack them into a single table and add the climate zone of each city.

In [11]:
ZONES= {
    "antananarivo" : "Central Highlands",
    "fianarantsoa" : "Southern Highlands",
    "toamasina" : "East coast",
    "mahajanga" : "West coast",
    "antsiranana" : "Far north",
    "toliara" : "Semi-arid south",
}

frames = [pd.read_csv(path, dtype={"date" : str}) for path in sorted(RAW_DIR.glob("*.csv"))]
df = pd.concat(frames, ignore_index = True)
df["zone"] = df["city"].map(ZONES)

print(df.shape)

(98616, 8)


In [12]:
print(df["city"].value_counts())
print("Cities without a zone: ", df["zone"].isna().sum())

city
antananarivo    16436
antsiranana     16436
fianarantsoa    16436
mahajanga       16436
toamasina       16436
toliara         16436
Name: count, dtype: int64
Cities without a zone:  0


## 2. Convert dates and check data types
Dates arrive as text (`19810101`). We convert them to real dates so we can extract years, months and seasons later.

In [13]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 98616 entries, 0 to 98615
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         98616 non-null  datetime64[us]
 1   city         98616 non-null  str           
 2   T2M          98616 non-null  float64       
 3   T2M_MAX      98616 non-null  float64       
 4   T2M_MIN      98616 non-null  float64       
 5   PRECTOTCORR  98616 non-null  float64       
 6   RH2M         98616 non-null  float64       
 7   zone         98616 non-null  str           
dtypes: datetime64[us](1), float64(5), str(2)
memory usage: 6.0 MB


## 3. Missing values
NASA POWER marks missing data with `-999`. We replace this code with a real missing value (`NaN`) so it can never distort averages, then count what is missing.

In [14]:
VALUE_COLS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M"]

n_fill = (df[VALUE_COLS] == -999).sum().sum()
df[VALUE_COLS] = df[VALUE_COLS].replace(-999, float("nan"))

print("-999 values replaced:", n_fill)
print(df[VALUE_COLS].isna().sum())

-999 values replaced: 0
T2M            0
T2M_MAX        0
T2M_MIN        0
PRECTOTCORR    0
RH2M           0
dtype: int64


## 4. Consistency checks
Each check counts the rows that break a rule that should always hold. All counts should be 0.

In [15]:
full_period = pd.date_range("1981-01-01", "2025-12-31", freq="D")

checks = {
    "Duplicate (city, date) rows": df.duplicated(["city", "date"]).sum(),
    "Missing days":                sum(len(full_period.difference(g["date"])) for _, g in df.groupby("city")),
    "T2M_MIN above T2M":           (df["T2M_MIN"] > df["T2M"]).sum(),
    "T2M above T2M_MAX":           (df["T2M"] > df["T2M_MAX"]).sum(),
    "Negative rainfall":           (df["PRECTOTCORR"] < 0).sum(),
    "Humidity outside 0–100 %":    (~df["RH2M"].between(0, 100)).sum(),
    "Temperature outside -5–50 °C": ((df["T2M_MIN"] < -5) | (df["T2M_MAX"] > 50)).sum(),
}
pd.Series(checks, name="Rows failing")

Duplicate (city, date) rows     0
Missing days                    0
T2M_MIN above T2M               0
T2M above T2M_MAX               0
Negative rainfall               0
Humidity outside 0–100 %        0
Temperature outside -5–50 °C    0
Name: Rows failing, dtype: int64

In [16]:
df[VALUE_COLS].describe().round(1)

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M
count,98616.0,98616.0,98616.0,98616.0,98616.0
mean,22.5,26.3,19.4,4.3,78.4
std,4.6,4.1,5.3,8.1,8.9
min,7.6,11.3,3.0,0.0,43.9
25%,19.4,23.6,15.5,0.2,72.6
50%,23.5,27.1,20.7,1.2,80.2
75%,26.1,29.1,23.8,4.8,85.4
max,33.5,39.8,29.2,244.1,95.5


In [17]:
for col, how in [("T2M_MAX", "idxmax"), ("T2M_MIN", "idxmin"), ("PRECTOTCORR", "idxmax")]:
    row = df.loc[getattr(df[col], how)()]
    print(f"{col:12} {row[col]:>6}  {row['city']:13} {row['date'].date()}")

T2M_MAX       39.84  toliara       2025-01-11
T2M_MIN        2.95  fianarantsoa  2009-06-30
PRECTOTCORR  244.12  antananarivo  2020-04-12


**Result:** all checks pass. No missing values, no duplicates, no missing days and no impossible values were found, so no rows were removed or modified.

In [18]:
mask = df["date"].between("2020-04-08", "2020-04-16")
df[mask].pivot(index="date", columns="city", values="PRECTOTCORR").round(1)

city,antananarivo,antsiranana,fianarantsoa,mahajanga,toamasina,toliara
date,,,,,,
2020-04-08,6.8,0.4,11.0,0.2,1.7,0.0
2020-04-09,3.8,5.2,10.0,0.1,5.2,0.1
2020-04-10,1.1,4.4,17.3,0.7,8.8,0.0
2020-04-11,50.0,0.7,9.9,29.2,64.8,0.0
2020-04-12,244.1,5.3,14.2,20.0,65.6,0.2
2020-04-13,95.4,5.6,6.3,55.8,83.3,0.0
2020-04-14,107.9,6.3,7.5,2.0,63.6,0.0
2020-04-15,147.0,2.6,8.9,4.6,82.3,0.2
2020-04-16,33.6,4.0,3.7,2.7,18.8,0.3


## 5. Add time columns
- `year`, `month`: extracted from the date.
- `season`: **rainy** (November–April) or **dry** (May–October).
- `hydro_year`: a "hydrological year" running from 1 July to 30 June, labelled by its starting year. The rainy season (November–April) crosses New Year; with `hydro_year`, each rainy season belongs to a single year instead of being split in two.

In [19]:
RAINY_MONTHS = [11, 12, 1, 2, 3, 4]

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["season"] = df["month"].isin(RAINY_MONTHS).map({True: "rainy", False: "dry"})
df["hydro_year"] = df["year"].where(df["month"] >= 7, df["year"] - 1)

df.groupby("hydro_year").size().iloc[[0, 1, -2, -1]]

hydro_year
1980    1086
1981    2190
2024    2190
2025    1104
dtype: int64

## 6. Save the clean dataset
Columns are reordered (identifiers first, then measurements) and rows are sorted by city and date. The raw files in `data/raw/` are never modified.

In [20]:
COLUMN_ORDER = ["date", "city", "zone", "year", "month", "season", "hydro_year", *VALUE_COLS]

df = df[COLUMN_ORDER].sort_values(["city", "date"]).reset_index(drop=True)

out_path = PROCESSED_DIR / "madagascar_climate_daily.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows × {df.shape[1]} columns to {out_path}")

Saved 98,616 rows × 12 columns to ../data/processed/madagascar_climate_daily.csv


In [21]:
check = pd.read_csv(out_path, parse_dates=["date"])
print(check.shape)
check.head(3)

(98616, 12)


,date,city,zone,year,month,season,hydro_year,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M
0,1981-01-01,antananarivo,Central Highlands,1981,1,rainy,1980,19.37,23.98,15.26,0.45,84.79
1,1981-01-02,antananarivo,Central Highlands,1981,1,rainy,1980,19.40,23.85,15.71,0.00,86.34
2,1981-01-03,antananarivo,Central Highlands,1981,1,rainy,1980,18.74,24.08,14.82,0.00,84.12
